In [48]:
import pandas as pd
import matplotlib.pyplot as plt
import google.oauth2.credentials
import google.oauth2.service_account
import pandas_gbq


## Extração de Dados
- Dados público obtidos na plataforma Kaggle
- Dados brutos no formato CSV com separdor em ','
- Processo de extração por meio da biblioteca Pandas

In [2]:
sales = pd.read_csv("../data/sales_car.csv", sep=",", encoding="utf-8")
sales.head()

,Car_id,Date,Customer Name,Gender,Annual Income,Dealer_Name,Company,Model,Engine,Transmission,Color,Price ($),Dealer_No,Body Style,Phone,Dealer_Region
0,C_CND_000001,1/2/2022,Geraldine,Male,13500,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,26000,06457-3834,SUV,8264678,Middletown
1,C_CND_000002,1/2/2022,Gia,Male,1480000,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,19000,60504-7114,SUV,6848189,Aurora
2,C_CND_000003,1/2/2022,Gianna,Male,1035000,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,31500,38701-8047,Passenger,7298798,Greenville
3,C_CND_000004,1/2/2022,Giselle,Male,13500,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,14000,99301-3882,SUV,6257557,Pasco
4,C_CND_000005,1/2/2022,Grace,Male,1465000,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,24500,53546-9427,Hatchback,7081483,Janesville


## Transformação do dados
- Limpeza e Padronização

In [3]:
# Visão geral dos dados
sales.info()

<class 'pandas.DataFrame'>
RangeIndex: 23906 entries, 0 to 23905
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Car_id         23906 non-null  str  
 1   Date           23906 non-null  str  
 2   Customer Name  23905 non-null  str  
 3   Gender         23906 non-null  str  
 4   Annual Income  23906 non-null  int64
 5   Dealer_Name    23906 non-null  str  
 6   Company        23906 non-null  str  
 7   Model          23906 non-null  str  
 8   Engine         23906 non-null  str  
 9   Transmission   23906 non-null  str  
 10  Color          23906 non-null  str  
 11  Price ($)      23906 non-null  int64
 12  Dealer_No      23906 non-null  str  
 13  Body Style     23906 non-null  str  
 14  Phone          23906 non-null  int64
 15  Dealer_Region  23906 non-null  str  
dtypes: int64(3), str(13)
memory usage: 2.9 MB


In [5]:
# Verificando duplicatas e valores nulos na coluna 'Car_id'
qte_duplicatas = sales['Car_id'].duplicated().sum()
n_nulls = sales['Car_id'].isnull().sum()

print(f"Quantidade de IDs duplicados: {qte_duplicatas}")
print(f"Quantidade de IDs nulos: {n_nulls}")

Quantidade de IDs duplicados: 0
Quantidade de IDs nulos: 0


In [6]:
# Renomeando colunas para português

rename_columns = {
    "Car_id": "car_id",
    "Date": "data_compra",
    "Customer Name": "nome_cliente",
    "Gender": "genero",
    "Annual Income": "renda_anual",
    "Dealer_Name": "concessionaria",
    "Company": "empresa",
    "Model": "modelo",
    "Engine": "motor",
    "Transmission": "transmissao",
    "Color": "cor",
    "Price ($)": "preco",
    "Dealer_No ": "numero_concessionaria",
    "Body Style": "estilo_carro",
    "Phone": "telefone",
    "Dealer_Region": "regiao_concessionaria"
}

sales.rename(columns=rename_columns, inplace=True)
sales.head()

,car_id,data_compra,nome_cliente,genero,renda_anual,concessionaria,empresa,modelo,motor,transmissao,cor,preco,numero_concessionaria,estilo_carro,telefone,regiao_concessionaria
0,C_CND_000001,1/2/2022,Geraldine,Male,13500,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,DoubleÂ Overhead Camshaft,Auto,Black,26000,06457-3834,SUV,8264678,Middletown
1,C_CND_000002,1/2/2022,Gia,Male,1480000,C & M Motors Inc,Dodge,Durango,DoubleÂ Overhead Camshaft,Auto,Black,19000,60504-7114,SUV,6848189,Aurora
2,C_CND_000003,1/2/2022,Gianna,Male,1035000,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,Red,31500,38701-8047,Passenger,7298798,Greenville
3,C_CND_000004,1/2/2022,Giselle,Male,13500,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,Pale White,14000,99301-3882,SUV,6257557,Pasco
4,C_CND_000005,1/2/2022,Grace,Male,1465000,Chrysler Plymouth,Acura,TL,DoubleÂ Overhead Camshaft,Auto,Red,24500,53546-9427,Hatchback,7081483,Janesville


In [7]:
#Substituindo valores nulos da coluna 'nome_cliente'
sales.fillna({'nome_cliente': 'Desconhecido'}, inplace=True)

,car_id,data_compra,nome_cliente,genero,renda_anual,concessionaria,empresa,modelo,motor,transmissao,cor,preco,numero_concessionaria,estilo_carro,telefone,regiao_concessionaria
7564,C_CND_007565,11/5/2022,Desconhecido,Male,680000,Saab-Belle Dodge,Dodge,Ram Pickup,DoubleÂ Overhead Camshaft,Auto,Pale White,45000,60504-7114,Hardtop,7203103,Aurora


In [8]:
# Corrigindo encoding da coluna motor
sales['motor'] = sales['motor'].str.encode('utf-8').str.decode('utf-8')
sales['motor'] = sales['motor'].str.replace('Â', '')
sales['motor']

0        Double Overhead Camshaft
1        Double Overhead Camshaft
2               Overhead Camshaft
3               Overhead Camshaft
4        Double Overhead Camshaft
                   ...           
23901           Overhead Camshaft
23902    Double Overhead Camshaft
23903           Overhead Camshaft
23904    Double Overhead Camshaft
23905    Double Overhead Camshaft
Name: motor, Length: 23906, dtype: str

In [23]:
# Transformando dados de data_compra para datetime no formato dd/mm/yyyy
sales['data_compra'] = pd.to_datetime(sales['data_compra'], format='%d/%m/%Y', errors='coerce')
sales['data_compra']

0       2022-01-02
1       2022-01-02
2       2022-01-02
3       2022-01-02
4       2022-01-02
           ...    
23901   2023-12-31
23902   2023-12-31
23903   2023-12-31
23904   2023-12-31
23905   2023-12-31
Name: data_compra, Length: 23906, dtype: datetime64[us]

In [24]:
# Transformando dados de renda_anual e preco para float
sales['renda_anual'] = sales['renda_anual'].astype(float)
sales['preco'] = sales['preco'].astype(float)

In [31]:
# Padronização de valores da coluna regiao_concessionaria, genero e cor
sales['regiao_concessionaria'] = sales['regiao_concessionaria'].str.upper()
sales['genero'] = sales['genero'].str[0].str.upper()
sales['cor'] = sales['cor'].str.upper()


In [32]:
sales.head()

,car_id,data_compra,nome_cliente,genero,renda_anual,concessionaria,empresa,modelo,motor,transmissao,cor,preco,numero_concessionaria,estilo_carro,telefone,regiao_concessionaria
0,C_CND_000001,2022-01-02,Geraldine,M,13500.0,Buddy Storbeck's Diesel Service Inc,Ford,Expedition,Double Overhead Camshaft,Auto,BLACK,26000.0,06457-3834,SUV,8264678,MIDDLETOWN
1,C_CND_000002,2022-01-02,Gia,M,1480000.0,C & M Motors Inc,Dodge,Durango,Double Overhead Camshaft,Auto,BLACK,19000.0,60504-7114,SUV,6848189,AURORA
2,C_CND_000003,2022-01-02,Gianna,M,1035000.0,Capitol KIA,Cadillac,Eldorado,Overhead Camshaft,Manual,RED,31500.0,38701-8047,Passenger,7298798,GREENVILLE
3,C_CND_000004,2022-01-02,Giselle,M,13500.0,Chrysler of Tri-Cities,Toyota,Celica,Overhead Camshaft,Manual,PALE WHITE,14000.0,99301-3882,SUV,6257557,PASCO
4,C_CND_000005,2022-01-02,Grace,M,1465000.0,Chrysler Plymouth,Acura,TL,Double Overhead Camshaft,Auto,RED,24500.0,53546-9427,Hatchback,7081483,JANESVILLE


In [33]:
sales.to_csv("../data/sales_car_tratada.csv", index=False, encoding="utf-8")

## Carregamento dos Dados
- Carregamento do dataset atraves de uma banco na nuvem (BigQuery) 
- Disponibilazação de dados para BI e dashboards

In [35]:
credentials = (google.oauth2.service_account
               .Credentials
               .from_service_account_file(filename="../config/GBQ.json", 
                                          scopes=["https://www.googleapis.com/auth/cloud-platform"])
               )

In [49]:
sales.to_gbq(destination_table="sales_analytics.car_sales", 
             if_exists="replace", 
             table_schema=[
                    {"name": "car_id", "type": "STRING"},
                    {"name": "data_compra", "type": "DATE"},
                    {"name": "nome_cliente", "type": "STRING"},
                    {"name": "genero", "type": "STRING"},
                    {"name": "renda_anual", "type": "FLOAT"},
                    {"name": "concessionaria", "type": "STRING"},
                    {"name": "empresa", "type": "STRING"},
                    {"name": "modelo", "type": "STRING"},
                    {"name": "motor", "type": "STRING"},
                    {"name": "transmissao", "type": "STRING"},
                    {"name": "cor", "type": "STRING"},
                    {"name": "preco", "type": "FLOAT"},
                    {"name": "numero_concessionaria", "type": "STRING"},
                    {"name": "estilo_carro", "type": "STRING"},
                    {"name": "telefone", "type": "STRING"},
                    {"name": "regiao_concessionaria", "type": "STRING"}],
             credentials=credentials)

AttributeError: 'DataFrame' object has no attribute 'to_gbq'